## Ingestion Transazioni — Auto Loader

In [0]:
CATALOG       = "czech_fintech"
BRONZE_SCHEMA = "bronze"
STORAGE_ROOT  = "abfss://raw@czechfintechdata.dfs.core.windows.net"
STATIC_PATH   = f"{STORAGE_ROOT}/static"
TRANS_PATH    = f"{STORAGE_ROOT}/transactions"
CHECKPOINT    = f"{STORAGE_ROOT}/checkpoints/bronze_transactions"

DEBUG

In [0]:
# # Drop table vecchia
# spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.transactions")

In [0]:
# # #DA USARE SOLO IN DEV TOGLIERLO DA PROD
# # Cancella checkpoint vecchio
# dbutils.fs.rm(CHECKPOINT, recurse=True)

# # # Oppure usane uno nuovo
# CHECKPOINT_NEW = f"{STORAGE_ROOT}/checkpoints/bronze_transactions_v2"

# # # Poi esegui l'ingest_transactions() con CHECKPOINT_NEW

====================


In [0]:
from pyspark.sql.types import StructType, StructField, StringType
import pyspark.sql.functions as F

CATALOG = "czech_fintech"
BRONZE_SCHEMA = "bronze"
STORAGE_ROOT = "abfss://raw@czechfintechdata.dfs.core.windows.net"
TRANS_PATH = f"{STORAGE_ROOT}/transactions"
CHECKPOINT = f"{STORAGE_ROOT}/checkpoints/bronze_transactions"

trans_schema = StructType([
    StructField("trans_id", StringType()),
    StructField("account_id", StringType()),
    StructField("date", StringType()),
    StructField("type", StringType()),
    StructField("operation", StringType()),
    StructField("amount", StringType()),
    StructField("balance", StringType()),
    StructField("k_symbol", StringType()),
    StructField("bank", StringType()),
    StructField("account", StringType()),
])

column_names = ["trans_id", "account_id", "date", "type", "operation", "amount", "balance", "k_symbol", "bank", "account"]

trans_bronze = (
    spark.readStream
    .text(TRANS_PATH)
    .filter(~F.col("value").startswith("trans_id"))
    .select(
        F.split(F.regexp_replace(F.col("value"), '"', ''), ";").alias("cols")
    )
    .select([F.col("cols")[i].alias(column_names[i]) for i in range(10)])
    .withColumn("_ingestion_ts", F.current_timestamp())
    .writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", CHECKPOINT)
    .toTable(f"{CATALOG}.{BRONZE_SCHEMA}.transactions")
)



spark.table(f'{CATALOG}.{BRONZE_SCHEMA}.transactions').filter(F.col("trans_id") != "trans_id").limit(5).show(truncate=False)

